# Reproduction source for embedded model demos

The lecture HTML contains the model demonstrations at the matching concepts. This notebook preserves the executable source and the original experiment sequence. Measured outputs and execution records are stored in `reviews/model-runtime/` and embedded into the lecture HTML by `tools/build.py`.

Prepare the pinned environment once with `bash tools/setup-model-env.sh`. Run the notebook with the **AX Model Labs** kernel. Show-o2 uses the separate **ax-showo2** runtime prepared by the same script. No package installation occurs during a classroom demonstration.

The HTML labels recorded inference separately from the small interactive calculations. Recorded responses are model outputs to inspect, not verified facts or new inference requests.


# 🎨 Lab 5: Multimodal Generation Practice
## Text-to-Image Generation → Native Unified Multimodal Generation (Show-o2)

This lab extends Lab 4 from **multimodal understanding** to **multimodal generation**.

In Lab 4, we used a multimodal LLM mainly for understanding images and answering questions.
In this lab, we study models that can **generate new outputs**.

### Goals
1. Use a text-to-image generation model to create images from prompts.
2. Use **Show-o2**, an open-source unified multimodal model, to connect image understanding and image generation.
3. Organize unified multimodal generation by application:
   - visual briefing
   - image-grounded prompt writing
   - visual response generation
   - multimodal assistant output

### Key idea
A native unified multimodal model is not just a VLM plus a separate image generator.
It is designed to support both **multimodal understanding** and **visual generation** within one model family.

> **Note.** Show-o2 is heavier than the models used in earlier labs. The Show-o2 cells are written for a Colab GPU runtime and include execution flags so that you can control which parts to run.


## 0. Environment Setup

In [ ]:
# One-time environment preparation: bash tools/setup-model-env.sh
# Installation is separate so running a lecture never upgrades a live kernel.
import importlib.metadata as metadata
import platform
intel_mac = platform.system() == "Darwin" and platform.machine() == "x86_64"
expected = {
    "torch": "2.2.2" if intel_mac else "2.7.1",
    "transformers": "4.57.1",
    "diffusers": "0.30.3",
    "timm": "1.0.22",
    "scikit-learn": "1.6.1",
    "tabpfn": "2.2.1",
    "chronos-forecasting": "2.3.2",
}
for package, wanted in expected.items():
    installed = metadata.version(package)
    if installed.split("+")[0] != wanted:
        raise RuntimeError(f"{package} {installed} does not match {wanted}. Run tools/setup-model-env.sh and restart the kernel.")
    print(f"{package} {installed}")


In [ ]:
import os, random, tarfile, requests, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   CPU execution is slower; the same model experiments will run.")

IMG_DIR = "./generation-pets"
CACHE_DIR = "./cache"
OUT_DIR = "./lab5_outputs"
for d in [IMG_DIR, CACHE_DIR, OUT_DIR, f"{IMG_DIR}/dog", f"{IMG_DIR}/cat"]:
    os.makedirs(d, exist_ok=True)

## 1. Data Preparation

We use a small subset of the Oxford-IIIT Pet dataset.

- In **Part A**, the dataset is only used as familiar visual context.
- In **Part B**, we select one pet image as the shared multimodal input for Show-o2 applications.

This makes the unified multimodal part easier to follow: the same image is repeatedly used for understanding, prompt writing, and image generation.


In [ ]:
NUM_IMAGES_PER_CLASS = 8

def download_oxford_pets():
    extract_dir = os.path.join(CACHE_DIR, "images")
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 100:
        print("   Using cached Oxford-IIIT Pet images.")
        return extract_dir

    url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
    tar_path = os.path.join(CACHE_DIR, "images.tar.gz")
    print(f"   Downloading: {url}")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(tar_path, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    with tarfile.open(tar_path, "r:gz") as t:
        t.extractall(CACHE_DIR)
    return extract_dir

def organize_pet_subset(src, dst, n):
    # Oxford-IIIT Pet naming convention:
    # cat breed files start with uppercase; dog breed files start with lowercase.
    dogs = sorted([f for f in os.listdir(src) if f.lower().endswith(".jpg") and f[0].islower()])
    cats = sorted([f for f in os.listdir(src) if f.lower().endswith(".jpg") and f[0].isupper()])
    random.shuffle(dogs)
    random.shuffle(cats)

    selected = []
    for label, files in [("dog", dogs[:n]), ("cat", cats[:n])]:
        out_dir = os.path.join(dst, label)
        os.makedirs(out_dir, exist_ok=True)
        for fname in files:
            src_path = os.path.join(src, fname)
            dst_path = os.path.join(out_dir, fname)
            if not os.path.exists(dst_path):
                Image.open(src_path).convert("RGB").save(dst_path)
            selected.append((dst_path, label))
    return selected

src_dir = download_oxford_pets()
pet_items = organize_pet_subset(src_dir, IMG_DIR, NUM_IMAGES_PER_CLASS)
image_paths = [p for p, y in pet_items]
image_labels = [y for p, y in pet_items]

print(f"✅ Prepared {len(image_paths)} images")
print(f"   Dog images: {sum(y == 'dog' for y in image_labels)}")
print(f"   Cat images: {sum(y == 'cat' for y in image_labels)}")

In [ ]:
def show_image_grid(images, titles=None, cols=4, figsize=(14, 7), suptitle=None):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(images):
            img = images[i]
            if isinstance(img, (str, Path)):
                img = Image.open(img).convert("RGB")
            ax.imshow(img)
            if titles is not None:
                ax.set_title(titles[i], fontsize=11, fontweight="bold")
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

n_show = min(8, len(image_paths))
show_image_grid(
    image_paths[:n_show],
    titles=[y.upper() for y in image_labels[:n_show]],
    cols=4,
    figsize=(14, 7),
    suptitle="🐶🐱 Sample Images for Lab 5",
)

---
# Part A: Text-to-Image Generation

## 2. What does a text-to-image diffusion model do?

A text-to-image model maps a **text prompt** into an **image**.

```text
Text prompt → Text encoder → Denoising process → Generated image
```

The model starts from random noise and gradually denoises it while being guided by the prompt.

```text
noise → rough structure → object shape → texture/color → final image
```

## 3. Load a Text-to-Image Model

In [ ]:
from diffusers import AutoPipelineForText2Image

# Fast default for practice. Good for short classroom experiments.
T2I_MODEL_ID = "stabilityai/sd-turbo"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"📦 Loading text-to-image model: {T2I_MODEL_ID}")
t2i_pipe = AutoPipelineForText2Image.from_pretrained(
    T2I_MODEL_ID,
    torch_dtype=dtype,
    variant="fp16" if torch.cuda.is_available() else None,
)
t2i_pipe = t2i_pipe.to(device)
if torch.cuda.is_available():
    try:
        t2i_pipe.enable_attention_slicing()
    except Exception:
        pass
print("✅ T2I model loaded!")

In [ ]:
def generate_t2i(prompt, negative_prompt=None, seed=0, num_inference_steps=2,
                 guidance_scale=0.0, height=512, width=512, save_name=None):
    generator_device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    start = time.time()
    result = t2i_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        generator=generator,
    )
    image = result.images[0]
    elapsed = time.time() - start
    if save_name:
        save_path = os.path.join(OUT_DIR, save_name)
        image.save(save_path)
        print(f"💾 Saved: {save_path}")
    return image, elapsed

## 4. Basic Prompt → Image Generation

In [ ]:
prompt = "A cute golden retriever puppy sitting in a cozy library, warm light, realistic photo"
image, elapsed = generate_t2i(
    prompt,
    seed=42,
    num_inference_steps=2,
    guidance_scale=0.0,
    save_name="basic_t2i_puppy.png",
)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f"Prompt → Image\n{elapsed:.1f}s", fontsize=12, fontweight="bold")
plt.axis("off")
plt.show()

print("📝 Prompt:")
print(prompt)

---
## 5. Prompt Engineering

A text-to-image prompt often contains several components.

| Component | Example |
|---|---|
| **Subject** | a dog, a cat, a robot |
| **Scene** | in a library, on a beach |
| **Style** | realistic photo, watercolor, 3D icon |
| **Composition** | close-up portrait, wide shot |
| **Lighting** | warm light, cinematic lighting |
| **Quality words** | detailed, high quality, sharp focus |

In [ ]:
base_subject = "a small cat wearing a tiny backpack"

prompt_variants = [
    base_subject,
    base_subject + ", realistic photo, sitting on a wooden desk",
    base_subject + ", watercolor illustration, soft pastel colors",
    base_subject + ", 3D animated character, studio lighting, white background",
]

generated_prompt_images = []
for i, p in enumerate(prompt_variants):
    img, _ = generate_t2i(
        p,
        seed=7,
        num_inference_steps=2,
        guidance_scale=0.0,
        save_name=f"prompt_variant_{i}.png",
    )
    generated_prompt_images.append(img)

show_image_grid(
    generated_prompt_images,
    titles=[f"Variant {i+1}" for i in range(len(prompt_variants))],
    cols=4,
    figsize=(18, 5),
    suptitle="🎨 Prompt Style Comparison",
)

for i, p in enumerate(prompt_variants):
    print(f"[Variant {i+1}] {p}")

---
## 6. Seed Control: Same Prompt, Different Images

The **seed** controls the starting noise.

- Same prompt + same seed → similar / reproducible image
- Same prompt + different seed → different image

In [ ]:
seed_prompt = "A fluffy white dog as an astronaut on the moon, cinematic lighting, realistic photo"
seeds = [0, 1, 2, 3]

seed_images = []
for s in seeds:
    img, _ = generate_t2i(
        seed_prompt,
        seed=s,
        num_inference_steps=2,
        guidance_scale=0.0,
        save_name=f"seed_{s}.png",
    )
    seed_images.append(img)

show_image_grid(
    seed_images,
    titles=[f"Seed = {s}" for s in seeds],
    cols=4,
    figsize=(18, 5),
    suptitle="🎲 Same Prompt, Different Seeds",
)

print(seed_prompt)

---
## 7. Inference Steps: Speed vs Detail

For a turbo model, even a few steps can produce an image.

In general:

- fewer steps → faster but less refined
- more steps → slower but sometimes more detailed

In [ ]:
step_prompt = "A corgi puppy wearing a red scarf, sitting in a snowy village, storybook style"
steps_list = [1, 2, 3, 4]

step_images, step_times = [], []
for steps in steps_list:
    img, elapsed = generate_t2i(
        step_prompt,
        seed=123,
        num_inference_steps=steps,
        guidance_scale=0.0,
        save_name=f"steps_{steps}.png",
    )
    step_images.append(img)
    step_times.append(elapsed)

show_image_grid(
    step_images,
    titles=[f"{s} step(s)\n{t:.1f}s" for s, t in zip(steps_list, step_times)],
    cols=4,
    figsize=(18, 5),
    suptitle="⏱️ Inference Step Comparison",
)

---
# Part B: Unified Multimodal Generation with Show-o2

In the previous version, the unified part mixed several models. Here, we use **Show-o2** as the main unified model.

Show-o2 is designed for native multimodal understanding and generation. In the official release, the practical demos are exposed through separate inference modes:

- **MMU / understanding mode:** `Image + Text → Text`
- **T2I / generation mode:** `Text → Image`
- **Mixed-modality generation:** conceptually `Image/Text → Text/Image`, but public demo support can vary by checkpoint and repo version

For this lab, we use one pet image as the shared input and organize the section by **applications**, not by model internals.

| Application | Input | Output | Why it matters |
|---|---|---|---|
| Visual Briefing | Image + Text | Text | Understand a visual input |
| Prompt Writing from Image | Image + Text | Structured Text | Convert image content into reusable generation instructions |
| Visual Response Generation | Text | Image | Generate a new visual output from the derived prompt |
| Multimodal Assistant Output | Image + Text | Text + Image | Use the same unified model workflow for both response types |

> The goal is to see what a unified multimodal model can do: it can understand an input image and then produce a generated visual response.


## 8. Select the Part B Reference Image

We select one real pet image from the dataset.
This image will be used throughout Part B.


In [ ]:
# Select one real pet image as the shared multimodal input.
PET_IMAGE_INDEX = 0

pet_image_path = image_paths[PET_IMAGE_INDEX]
pet_label = image_labels[PET_IMAGE_INDEX]
pet_image = Image.open(pet_image_path).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(pet_image)
plt.axis('off')
plt.title(f"Reference pet image: {pet_label}")
plt.show()

print("Reference image path:", pet_image_path)
print("Ground-truth label:", pet_label)


## 9. Set Up Show-o2

Show-o2 is an open-source unified multimodal model from Show Lab.
The official repository provides scripts for multimodal understanding and text-to-image generation.

This section is written for a GPU runtime. On a CPU runtime, the cells will be skipped.

### Practical notes
- Use a Colab GPU runtime.
- The first run downloads large model files.
- The official Show-o2 scripts use Weights & Biases logging by default, so we disable online logging with `WANDB_MODE=disabled`.
- The helper cells below patch the official scripts slightly so outputs are also saved/printed locally inside the notebook.


In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

RUN_SHOWO2 = True
SHOWO_REVISION = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"
SHOWO_ROOT = Path.home() / ".cache" / "ax-model-labs" / "Show-o"
SHOWO2_DIR = SHOWO_ROOT / "show-o2"
showo_env = Path.home() / ".venvs" / "ax-showo2" / "bin" / "python"
SHOWO_PYTHON = str(showo_env) if showo_env.exists() else sys.executable
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Show-o2 runtime:", SHOWO_PYTHON)
print("Show-o2 device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
if not SHOWO_ROOT.exists():
    SHOWO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/showlab/Show-o.git", str(SHOWO_ROOT)], check=True)
subprocess.run(["git", "-C", str(SHOWO_ROOT), "checkout", SHOWO_REVISION], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(SHOWO_ROOT), "rev-parse", "HEAD"], text=True).strip()
assert actual_revision == SHOWO_REVISION
print("Pinned official Show-o2 source:", actual_revision)


In [ ]:
# Prepare the dedicated runtime with tools/setup-model-env.sh before the class.
# Do not install or upgrade packages in a running kernel.
subprocess.run([SHOWO_PYTHON, "-c", "import torch, transformers, diffusers, timm, wandb; print(torch.__version__, transformers.__version__, diffusers.__version__)"], check=True)
from huggingface_hub import hf_hub_download
vae_checkpoint = hf_hub_download("Wan-AI/Wan2.1-T2V-14B", "Wan2.1_VAE.pth")
SHOWO_VAE = Path(vae_checkpoint).resolve()
print("Wan2.1 VAE is ready.")


## 10. Prepare Local Output Helpers

The official scripts are mainly written as demo scripts.
For a lab notebook, it is helpful to save results locally.

We prepare:
- a prompt file for Show-o2 text-to-image generation
- a local output directory
- a small patch so generated images are saved to files
- a small patch so understanding responses are printed in the notebook


In [ ]:
SHOWO2_OUTPUT_DIR = Path("./showo2_outputs")
SHOWO2_OUTPUT_DIR.mkdir(exist_ok=True)

# The official T2I script reads prompts from a text file.
PROMPT_FILE = Path("./showo2_prompts.txt")

print("Output directory:", SHOWO2_OUTPUT_DIR.resolve())
print("Prompt file:", PROMPT_FILE.resolve())


In [ ]:
#!/usr/bin/env python3
"""Adapt the pinned official image inference scripts to the classroom CPU runtime.

Attention math and trained weights stay upstream. CPU uses the official dense
attention masks; FlexAttention entry points fail explicitly if accidentally used.
"""
import argparse
from pathlib import Path
import subprocess

REVISION = '45a5a2de01d1ebd10cd5864d29310a76476cdf23'


def prepare(root):
    revision = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    if revision != REVISION:
        raise RuntimeError(f'Expected Show-o revision {REVISION}, found {revision}')
    folder = root / 'show-o2'
    compatibility = '''try:
    from torch.nn.attention.flex_attention import BlockMask, flex_attention, create_block_mask
except ImportError:
    class BlockMask:
        pass
    def flex_attention(*args, **kwargs):
        raise RuntimeError("FlexAttention requires a newer PyTorch. Use the dense inference path on CPU.")
    create_block_mask = flex_attention
'''
    (folder / 'ax_attention_compat.py').write_text(compatibility)
    for name in ['models/qwen2.py', 'models/modules.py', 'models/omni_attention.py',
                 'models/modeling_showo2_qwen2_5.py', 'inference_mmu.py', 'inference_t2i.py']:
        path = folder / name
        source = path.read_text()
        source = source.replace('wandb.util.generate_id()', '__import__("uuid").uuid4().hex[:8]')
        source = source.replace('from torch.nn.attention.flex_attention import flex_attention, create_block_mask',
                                'from ax_attention_compat import flex_attention, create_block_mask')
        source = source.replace('from torch.nn.attention.flex_attention import BlockMask',
                                'from ax_attention_compat import BlockMask')
        if name == 'inference_mmu.py':
            marker = "        responses[j] += f'User: ' + question + f'\\n Answer : ' + text[0] + '\\n'"
            if '# AX_RESPONSE_JSON' not in source:
                if marker not in source:
                    raise RuntimeError('The upstream MMU output anchor changed.')
                source = source.replace(marker, marker + '\n        import json\n        print("AX_RESPONSE_JSON:" + json.dumps(text[0]), flush=True)  # AX_RESPONSE_JSON')
            source = source.replace('max_new_tokens=300,', 'max_new_tokens=config.get("max_new_tokens", 96),')
            if '    set_seed(config.get("seed", 42))' not in source:
                source = source.replace('    # VQ model for processing image into discrete tokens', '    set_seed(config.get("seed", 42))\n\n    # VQ model for processing image into discrete tokens')
        if name == 'inference_t2i.py':
            source = source.replace(')).to(torch.bfloat16).to(device)', ')).to(weight_type).to(device)')
            if '# AX_SEED' not in source:
                source = source.replace('    config = get_config()', '    config = get_config()\n    set_seed(config.get("seed", 42))  # AX_SEED')
            marker = '        wandb_images = [wandb.Image(image, caption=prompts[i]) for i, image in enumerate(checked_images)]'
            if '# AX_SAVE_IMAGES' not in source:
                if marker not in source:
                    raise RuntimeError('The upstream T2I output anchor changed.')
                source = source.replace(marker, '''        # AX_SAVE_IMAGES
        os.makedirs(config.experiment.output_dir, exist_ok=True)
        for image_index, generated_image in enumerate(checked_images):
            generated_image.save(os.path.join(config.experiment.output_dir, f"sample_{step + image_index:03d}.png"))
''' + marker)
        if name in ['inference_mmu.py', 'inference_t2i.py'] and '# AX_CPU_INT8' not in source:
            source = source.replace('    model.eval()', '''    model.eval()
    # AX_CPU_INT8: dynamic quantization of linear layers for the free CPU runtime.
    if device.type == "cpu" and config.get("cpu_int8", True):
        torch.backends.quantized.engine = "fbgemm"
        model = torch.ao.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True)
        print("CPU linear weights: dynamic int8; other operations: float32", flush=True)''')
        path.write_text(source)
    path = folder / 'utils.py'
    source = path.read_text()
    if '# Image-only classroom runtime' not in source:
        source = source.replace('from decord import VideoReader, cpu',
            'try:\n    from decord import VideoReader, cpu\nexcept ImportError:\n    VideoReader = cpu = None  # Image-only classroom runtime')
    path.write_text(source)
    return folder



prepare(SHOWO_ROOT)
print("Pinned image inference scripts are ready.")


## 11. Application 1 — Visual Briefing

**Application goal:** use the pet image as visual context and generate a short text briefing.

```text
Pet image + instruction → text response
```

This is the understanding side of the unified model.


In [ ]:
briefing_question = (
    "Describe this pet image as a visual briefing for an image-generation model. "
    "Mention the animal, appearance, pose, background, and mood. "
    "Keep it concise."
)

print("Question for Show-o2 MMU:")
print(briefing_question)


In [ ]:
def run_showo2_mmu(image_path, question, config_name="configs/showo2_1.5b_demo_432x432.yaml"):
    cmd = [SHOWO_PYTHON, "inference_mmu.py", f"config={config_name}",
        f"mmu_image_path={Path(image_path).resolve()}", f"question={question}",
        f"model.vae_model.pretrained_model_path={SHOWO_VAE}", "max_new_tokens=160", "seed=42"]
    started = time.perf_counter()
    result = subprocess.run(cmd, cwd=SHOWO2_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    answers = [line.split("AX_RESPONSE_JSON:", 1)[1] for line in result.stdout.splitlines() if line.startswith("AX_RESPONSE_JSON:")]
    if not answers:
        raise RuntimeError("Show-o2 returned no structured response.\n" + result.stdout[-3000:])
    answer = json.loads(answers[-1])
    if not answer.strip():
        raise RuntimeError("Show-o2 returned an empty answer.")
    print("Question:", question)
    print("Response:", answer)
    print(f"Measured MMU call: {time.perf_counter()-started:.1f}s")
    return answer

briefing_text = run_showo2_mmu(pet_image_path, briefing_question)


## 12. Application 2 — Prompt Writing from Image

**Application goal:** convert the pet image into a reusable image-generation prompt.

```text
Pet image + prompt-writing request → structured text prompt
```

This is useful because many generation tasks begin by turning a visual reference into a clean prompt.


In [ ]:
prompt_question = (
    "Based on this pet image, write a clean text-to-image prompt. "
    "Use this format only: Subject: ... Style: ... Scene: ... Details to preserve: ..."
)

print(prompt_question)


In [ ]:
image_grounded_prompt_text = run_showo2_mmu(pet_image_path, prompt_question)


In [ ]:
# Pass the actual model response to generation. No manual success substitute.
showo2_pet_prompt = image_grounded_prompt_text.strip()
assert showo2_pet_prompt
print("Actual image-grounded prompt:")
print(showo2_pet_prompt)


## 13. Application 3 — Visual Response Generation

**Application goal:** generate a new image from the image-grounded prompt.

```text
Text prompt → image
```

This is the generation side of Show-o2.
We use the same unified model family, but switch to the visual generation mode.


In [ ]:
SHOWO_CALL_NUMBER = 0

def run_showo2_t2i(prompts, config_name="configs/showo2_1.5b_demo_432x432.yaml", guidance_scale=5.0, num_inference_steps=25, batch_size=1):
    global SHOWO_CALL_NUMBER
    if isinstance(prompts, str):
        prompts = [prompts]
    if not prompts or any(not prompt.strip() for prompt in prompts):
        raise ValueError("Generation requires a non-empty prompt.")
    SHOWO_CALL_NUMBER += 1
    import uuid
    out_dir = (SHOWO2_OUTPUT_DIR / f"call_{SHOWO_CALL_NUMBER:02d}_{uuid.uuid4().hex[:8]}").resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    prompt_path = out_dir / "prompts.txt"
    prompt_path.write_text("\n".join(prompts))
    cmd = [SHOWO_PYTHON, "inference_t2i.py", f"config={config_name}",
        f"dataset.params.validation_prompts_file={prompt_path}",
        f"experiment.output_dir={out_dir}", f"guidance_scale={guidance_scale}",
        f"num_inference_steps={num_inference_steps}", f"batch_size={batch_size}",
        f"model.vae_model.pretrained_model_path={SHOWO_VAE}",
        "model.weight_type=float32", "seed=42"]
    started = time.perf_counter()
    result = subprocess.run(cmd, cwd=SHOWO2_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    generated = sorted(out_dir.glob("sample_*.png"))
    if len(generated) != len(prompts):
        raise RuntimeError(f"Expected {len(prompts)} images, found {len(generated)}.\n" + result.stdout[-3000:])
    print("Actual generation prompts:", prompts)
    print(f"Measured T2I call: {time.perf_counter()-started:.1f}s")
    return generated

showo2_generated_paths = run_showo2_t2i(showo2_pet_prompt, num_inference_steps=25)


In [ ]:
# Display generated Show-o2 image(s), if available.
if showo2_generated_paths:
    imgs = [Image.open(p).convert('RGB') for p in showo2_generated_paths[:4]]
    titles = [p.name for p in showo2_generated_paths[:4]]
    show_image_grid(imgs, titles=titles, cols=len(imgs), figsize=(12, 4), suptitle="Show-o2 generated image(s)")
else:
    print("No Show-o2 image files found yet. Run the T2I cell on a GPU runtime.")


## 14. Application 4 — Multimodal Assistant Output

**Application goal:** produce both a text answer and a visual response for the same user request.

```text
Pet image + user request → text plan + generated image
```

This is the clearest classroom way to show `Image + Text` input and `Text + Image` output using Show-o2:

1. Show-o2 reads the image and the request to produce a text plan.
2. Show-o2 uses the plan as a visual-generation prompt to produce an image.

Both steps use Show-o2, but the public demo interface exposes them as separate inference modes.


In [ ]:
assistant_request = (
    "Use this pet as a children's book character. Reply with three short lines: "
    "TITLE: a title of up to six words. CONCEPT: one short sentence. "
    "IMAGE_PROMPT: one text-to-image prompt of at most fifty words preserving the pet's visible appearance."
)
assistant_text = run_showo2_mmu(pet_image_path, assistant_request)


In [ ]:
if "IMAGE_PROMPT:" not in assistant_text:
    raise ValueError("The model did not produce the required IMAGE_PROMPT field. Inspect the actual response above.")
assistant_visual_prompt = assistant_text.split("IMAGE_PROMPT:", 1)[1].strip()
assert assistant_visual_prompt
assistant_image_paths = run_showo2_t2i(assistant_visual_prompt, num_inference_steps=25)
img = Image.open(assistant_image_paths[0]).convert('RGB')
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title("Show-o2 assistant image from the actual text response")
plt.show()


## 15. Optional — Direct Mixed-Modality Generation

Some unified multimodal models can produce interleaved outputs such as:

```text
input:  image + text
output: text + image + text
```

Show-o2 is designed for mixed-modality generation, but the exact public inference script can depend on the released checkpoint and repository version.
The cell below checks whether the cloned repository includes an official mixed-modality inference script.

If such a script is available in your version of the repo, you can adapt it here.
If not, use Application 4 above as the stable lab version: **same model family, image-grounded text planning, then visual generation**.


In [ ]:
if SHOWO2_DIR.exists():
    possible_mixed_scripts = sorted(SHOWO2_DIR.glob("*mixed*.py")) + sorted(SHOWO2_DIR.glob("*interleave*.py"))
    if possible_mixed_scripts:
        print("Possible mixed-modality scripts found:")
        for p in possible_mixed_scripts:
            print("-", p)
    else:
        print("No obvious mixed-modality inference script found in this repo snapshot.")
        print("Use Application 4 as the stable Show-o2 workflow for this lab.")
else:
    print("Show-o2 repository is not cloned yet.")


## 16. Compare the Applications

| Application | Uses reference pet image? | Input | Output | Show-o2 role |
|---|---:|---|---|---|
| Visual Briefing | Yes | Image + Text | Text | Understanding |
| Prompt Writing from Image | Yes | Image + Text | Structured Text | Understanding + instruction following |
| Visual Response Generation | Indirectly | Text | Image | Generation |
| Multimodal Assistant Output | Yes | Image + Text | Text + Image | Understanding + generation |

### What changed from Lab 4?

Lab 4 focused on **understanding**:

```text
Image + Text → Text
```

Lab 5 focuses on **generation**:

```text
Text → Image
Image + Text → Text
Image + Text → Text + Image
```

The important shift is that the model output is no longer only a caption or answer.
It can also be a newly generated visual artifact.


In [ ]:
application_summary = {
    "Visual Briefing": {
        "Input": "Pet image + instruction",
        "Output": "Text briefing",
        "Show-o2 mode": "MMU / understanding",
    },
    "Prompt Writing from Image": {
        "Input": "Pet image + prompt-writing request",
        "Output": "Structured text prompt",
        "Show-o2 mode": "MMU / understanding",
    },
    "Visual Response Generation": {
        "Input": "Text prompt",
        "Output": "Generated image",
        "Show-o2 mode": "T2I / generation",
    },
    "Multimodal Assistant Output": {
        "Input": "Pet image + user request",
        "Output": "Text plan + generated image",
        "Show-o2 mode": "MMU + T2I",
    },
}

print(json.dumps(application_summary, indent=2, ensure_ascii=False))


---
## 📝 Lab 1~5 Grand Summary

### Per-Model Comparison

| Lab | Model | Paradigm | Input | Output | Core ability |
|-----|------|---------|------|--------|--------------|
| Lab 1 | CNN | Supervised learning | Image | Class label | Visual recognition |
| Lab 2 | ViT | Transformer vision model | Image patches | Class label | Patch-based recognition |
| Lab 3 | CLIP | Contrastive vision-language model | Image + candidate texts | Similarity score | Zero-shot classification |
| Lab 4 | LLM / VLM | Autoregressive multimodal understanding | Text or Image+Text | Text | Language and visual reasoning |
| Lab 5-A | Text-to-Image diffusion | Generative image model | Text | Image | Prompt-based image synthesis |
| Lab 5-B | Show-o2 | Unified multimodal model | Image + Text | Text + Image | Understanding and generation |

### Final takeaway

Earlier labs ask:

> Can the model recognize or describe an image?

Lab 5 asks:

> Can the model create a new image from language or from multimodal context?

This is the key transition from **perception** to **generation**.
